In [1]:
import numpy as np
import pyvista as pv


def plot_data(
        data: np.ndarray | list,
        size: float = 5.0,
):
    coords = data
    colors = np.clip(data, 0, 1)

    cloud = pv.PolyData(coords)
    cloud["colors"] = (colors * 255).astype(np.uint8)

    plotter = pv.Plotter()
    plotter.add_points(
        cloud,
        scalars="colors",
        rgb=True,
        point_size=size
    )

    plotter.add_axes()
    plotter.show_grid()
    plotter.show()


def plot_clustered_pv(
        data: np.ndarray,
        labels: np.ndarray,
        cluster_centers: np.ndarray,
        size: float = 4.0
):
    coords = data
    cluster_colors = cluster_centers[labels]
    colors = np.clip(cluster_colors, 0, 1)

    cloud = pv.PolyData(coords)
    cloud["colors"] = (colors * 255).astype(np.uint8)

    plotter = pv.Plotter()
    plotter.add_points(
        cloud,
        scalars="colors",
        rgb=True,
        point_size=size
    )

    plotter.add_axes()
    plotter.show_grid()
    plotter.show()


def plot_centroids_pv(
        cluster_centers: np.ndarray,
        size: float = 18.0
):
    coords = cluster_centers
    colors = np.clip(cluster_centers, 0, 1)

    cloud = pv.PolyData(coords)
    cloud["colors"] = (colors * 255).astype(np.uint8)

    plotter = pv.Plotter()
    plotter.add_points(
        cloud,
        scalars="colors",
        rgb=True,
        point_size=size,
        render_points_as_spheres=True
    )

    plotter.add_axes()
    plotter.show_grid()
    plotter.show()

In [2]:
import numpy as np

def generate_dataset(centers, stds, n_samples=500, ndim=3):
    X = []
    y = []

    for i, center in enumerate(centers):
        cluster = np.random.normal(loc=center, scale=stds[i], size=(n_samples // len(centers), ndim))
        X.append(cluster)
        y.append(np.full(n_samples // len(centers), i))

    return np.vstack(X), np.concatenate(y)

In [3]:
import pandas as pd
import pyvista as pv
import matplotlib.pyplot as plt

default_colors = np.array(plt.colormaps.get_cmap('tab10').colors)

In [4]:
data, ground_truth = generate_dataset(
    [[1, 1, 1], [3, 3, 3], [2, 2, 1]],
    stds=[.5, .5, .5],
    n_samples=500
)

pd.DataFrame(data)

,0,1,2
0,1.506722,1.681216,0.463116
1,1.604329,1.253445,1.117087
2,0.509027,1.223710,1.888333
3,0.822344,0.782306,1.604476
4,0.848094,0.228359,-0.093830
...,...,...,...
493,1.941741,2.656070,0.882226
494,2.149210,3.086880,0.814791
495,2.015622,2.503334,0.646113
496,2.969026,2.213026,1.483754


In [5]:
plt = pv.Plotter()

for n, d in zip(ground_truth, data):
    cloud = pv.PolyData(d)
    color = default_colors[n % len(default_colors)]

    plt.add_points(
        cloud,
        color=color,
        render_points_as_spheres=True,
        smooth_shading=True,
        point_size=10
    )

plt.add_axes()
plt.show_grid()
plt.show()

Widget(value='<iframe src="http://localhost:42689/index.html?ui=P_0x7fc3b10e6a50_0&reconnect=auto" class="pyvi…

In [25]:
from ktree.ntree import NTreeDynamic

tree = NTreeDynamic(1)

for a in data:
    tree.insert(a)

sorted_data = tree.sort()

In [46]:
n_clusters = 3

all_clusters = sorted(sorted_data, key=lambda x: len(x))[::-1]
clusters = all_clusters[:n_clusters]
all_data = all_clusters[n_clusters:]

for c in clusters:
    print(c.shape)
    for data in all_data:
        data = np.array([*data])
        # np.linalg.norm(p1 - p2)

        print(data)


[[0.013629391852868489, 2.1782431707265912], [-0.2623841958214983, 2.1104108449041683], [-0.35870935045877217, 2.1249917148079547]]
[[ 3.12832814  3.18872999  2.11944335]
 [ 3.03159854  2.73062998  2.00990897]
 [ 2.78434242  2.88169839  2.15125544]
 [ 3.18040582  2.83447547  2.1069155 ]
 [ 3.18719969  3.40649409  2.07236305]
 [ 2.34446542  2.72115589  1.59765929]
 [ 2.83750189  3.49072284  2.02273463]
 [ 2.77039655  3.55627419  2.11007632]
 [ 2.80336069  2.16228403  1.93071528]
 [ 2.74716655  2.99349008  1.71594128]
 [ 2.41820597  2.65308188  1.98589881]
 [ 3.26176388  3.3825742   2.08686319]
 [ 2.58133     2.34671606  1.5380943 ]
 [ 2.45362572  2.64779467  1.316577  ]
 [ 2.6840621   3.09144603  0.82613955]
 [ 2.25976098  2.4065506   1.76741148]
 [ 2.34549679  2.14649261  0.54254061]
 [ 2.84651536  2.6967202   0.90971441]
 [ 2.35486948  2.97191432  1.21862742]
 [ 2.33122044  2.85564241  0.57204026]
 [ 2.30937377  2.14109372  0.35598858]
 [ 2.43916739  2.28609019  0.4731002 ]
 [ 2.43210

In [39]:
plt = pv.Plotter()

for n, cluster in enumerate(sorted_data):
    s_data = list(cluster)
    cloud = pv.PolyData(s_data)
    color = default_colors[n % len(default_colors)]

    centroid = np.mean(s_data, axis=0)

    plt.add_points(
        cloud,
        color=color,
        render_points_as_spheres=True,
        smooth_shading=True,
        point_size=2
    )

    plt.add_points(
        centroid,
        color=color,
        render_points_as_spheres=True,
        smooth_shading=True,
        point_size=len(s_data) // 2
    )


plt.add_axes()
plt.show_grid()
plt.show()

Widget(value='<iframe src="http://localhost:42689/index.html?ui=P_0x7fc3a9163ed0_17&reconnect=auto" class="pyv…